# Packages

In [ ]:
!pip install swig
!pip install "gymnasium[box2d]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 71.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 24.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for box2d-py: filename=box2d_py-2.3.5-cp312-cp312-linux_x86_64.whl size=2399015 sha256=e0406a8e3a29f14f1e2d6be4063b9e23e3fc5ff605c6e9a898563511ab97fb1d
  Stored in directory: /root/.cache/pip/wheels/2a/e9/60/774da0bcd07f7dc7761a8590fa2d065e4069568e78dcdc3318
Successfully built box2d-py


In [ ]:
!pip install gym

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random


In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Replay Buffer

In [ ]:
transition=namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))

In [ ]:
class Buffer(object):
  def __init__(self, capacity):
    self.capacity=capacity
    self.memory=[]
    self.position=0

  def push(self, batch):
    self.memory.append(batch)
    if len(self.memory)>self.capacity:
      del self.memory[0]

  def sample(self, batch_size):
    return random.sample(self.memory, batch_size)

  def __len__(self):
    return len(self.memory)



# Model

In [ ]:
class QNetwork(nn.Module):
  def __init__(self, input_dim, output_dim, hidden_dim):
    super().__init__()

    self.linear1=nn.Linear(input_dim, hidden_dim)
    self.linear2=nn.Linear(hidden_dim, hidden_dim)
    self.linear3=nn.Linear(hidden_dim, hidden_dim)
    self.op=nn.Linear(hidden_dim, output_dim)


    self.act1=nn.ReLU()
    self.act2=nn.ReLU()
    self.act3=nn.ReLU()

  def forward(self, x):
    x=self.linear1(x)
    x=self.act1(x)
    x=self.linear2(x)
    x=self.act2(x)
    x=self.linear3(x)
    x=self.act3(x)
    x=self.op(x)
    return x


# LunarLander

In [ ]:
class LunarLander():
  lr=0.001
  replay_mem_size=1000
  mini_batch_size=32
  loss=nn.MSELoss()
  optim=None
  network_sync_rate=10 #No of steps the agent takes before syncing both target and policy network
  gamma=0.9

  def optimized_episode(self, mini_batch, policy_dqn, target_dqn):
    states, actions, rewards, new_states, terminations=zip(*mini_batch)
    ## Stack for faster calc
    states=torch.stack([torch.FloatTensor(s) for s in states]).to(device)
    actions=torch.tensor(actions).unsqueeze(1).to(device)
    new_states=torch.stack([torch.FloatTensor(s) for s in new_states]).to(device)
    rewards=torch.tensor(rewards, dtype=torch.float).unsqueeze(1).to(device)
    terminations=torch.tensor(terminations, dtype=torch.float).unsqueeze(1).to(device)

    with torch.no_grad():
      target_q_vals_next=target_dqn(new_states)
      max_target_q_vals, _=target_q_vals_next.max(dim=1, keepdim=True)
      max_target_q_val=self.gamma*max_target_q_vals
      final_future_val=(1-terminations)*max_target_q_val
      bellman_target=rewards+final_future_val

    current_q_estimate=policy_dqn(states).gather(1, actions)
    loss=self.loss(current_q_estimate, bellman_target)
    self.optim.zero_grad()
    loss.backward()
    self.optim.step()
    return loss.item()

  def train(self, episodes, render=False):
    env=gym.make('LunarLander-v3', render_mode='human' if render else None)
    num_states=env.observation_space.shape[0]
    num_actions=env.action_space.n
    epsilon=0.1
    memory=Buffer(self.replay_mem_size)
    self.loss_history=[]

    self.policy_qn=QNetwork(num_states, num_actions, 64).to(device)
    target_qn=QNetwork(num_states, num_actions, 64).to(device)
    target_qn.load_state_dict(self.policy_qn.state_dict()) #Intialise both with same weights

    self.optim=optim.Adam(self.policy_qn.parameters(), lr=self.lr)

    eps_decay=0.995
    min_eps=0.01

    steps=0

    for i in range(episodes):
      state,_=env.reset()
      terminated=False
      truncated=False
      while(not terminated and not truncated):
        rand=random.random()
        if rand<epsilon:
          action=env.action_space.sample()
        else:
          with torch.no_grad():
            state_tensor=torch.FloatTensor(state).to(device)

            action=self.policy_qn(state_tensor).argmax().item()

        new_state, reward, terminated, truncated,_=env.step(action)
        memory.push((state, action, reward, new_state, terminated))

        states=new_state
        steps+=1
        if len(memory)>self.mini_batch_size:
          mini_batch=memory.sample(self.mini_batch_size)
          loss_val=self.optimized_episode(mini_batch, self.policy_qn, target_qn)
          self.loss_history.append(loss_val)

        if steps>=self.network_sync_rate:
          target_qn.load_state_dict(self.policy_qn.state_dict())
          steps=0

        #Eps Decay
        epsilon=max(epsilon*eps_decay, min_eps)
      if i%10==0:
          print(f"Episode {i}, Epsilon{epsilon:.2f}, loss:{loss_val}")






In [ ]:
if __name__ == '__main__':
    agent = LunarLander()
    agent.train(episodes=100, render=False)

Episode 0, Epsilon0.01, loss:10.284000396728516
Episode 10, Epsilon0.01, loss:1.977803349494934
Episode 20, Epsilon0.01, loss:2.401695728302002
Episode 30, Epsilon0.01, loss:2.0242526531219482
Episode 40, Epsilon0.01, loss:0.4164966940879822
Episode 50, Epsilon0.01, loss:0.37715691328048706
Episode 60, Epsilon0.01, loss:1.7891887426376343
Episode 70, Epsilon0.01, loss:0.4372211694717407
Episode 80, Epsilon0.01, loss:0.39425402879714966
Episode 90, Epsilon0.01, loss:0.5053713321685791


# Testing

In [ ]:
import imageio
import numpy as np

def save_gif(agent, filename="lunar_lander.gif"):
    # 1. Setup environment for visual recording
    env = gym.make('LunarLander-v3', render_mode='rgb_array')

    state, _ = env.reset()
    done = False
    frames = []

    # 2. Run the game
    while not done:
        # Capture the frame
        frame = env.render()
        frames.append(frame)

        # Select Action (Pure Exploitation - No Randomness)
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            # Use the policy network directly
            action = agent.policy_qn(state_tensor).argmax().item()

        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    env.close()

    # 3. Save frames as GIF
    print(f"Saving {len(frames)} frames to {filename}...")
    imageio.mimsave(filename, frames, fps=30)
    print("Done!")


In [ ]:
save_gif(agent)

Saving 598 frames to lunar_lander.gif...
Done!
